<a href="https://colab.research.google.com/github/tsilva/aiml-notebooks/blob/main/rnn/wip-rnn-bit-parity-classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RNN - Bit-parity classifier

- TODO: add plotting
- TODO: make sure test set improves
- TODO: precreate set
- TODO: add variable length support

In this notebook we'll try to build an RNN to memorize how to classify bit sequences and classify their parity. It should output `1` if it has an odd number of `1`s and `0` if they are even.

In [30]:
def setup_config():
    #@markdown Random seed for reproducibility
    seed = 42  # @param {type:"integer"}

    #@markdown Number of training epochs
    n_epochs = 1000 # @param {type:"integer"}

    #@markdown Batch size
    batch_size = 64  # @param {type:"integer"}

    #@markdown Learning rate for the optimizer
    learning_rate = 0.0005  # @param {type:"number"}

    #@markdown Length of the input sequence (number of time steps)
    sequence_length = 10  # @param {type:"integer"}

    #@markdown Number of hidden units in the RNN
    hidden_size = 64  # @param {type:"integer"}

    #@markdown Activation function to use in the RNN ('tanh' or 'relu')
    nonlinearity = 'tanh'  # @param ['tanh', 'sigmoid', 'relu']

    #@markdown Weight initialization strategy ('none', 'xavier', or 'kaiming')
    weight_init = 'xavier'  # @param ['none', 'xavier', 'kaiming']

    #@markdown Maximum gradient norm for clipping (0.0 means no clipping)
    max_grad_norm = 5.0  # @param {type:"number"}

    # Calculate train/val split sizes (e.g., 80% train, 20% val)
    train_size = 0.8
    val_size = 0.2

    # These are meant to be hardcoded in this notebook
    batch_size = 32
    input_size = 1
    output_size = 2

    return {
        'seed': seed,
        'n_epochs': n_epochs,
        'batch_size': batch_size,
        'learning_rate': learning_rate,
        'sequence_length': sequence_length,
        'input_size': input_size,
        'hidden_size': hidden_size,
        'output_size': output_size,
        'nonlinearity': nonlinearity,
        'weight_init': weight_init,
        'max_grad_norm': max_grad_norm,
        'train_size': train_size,
        'val_size': val_size
    }

CONFIG = setup_config()

Set manual seed for reproducibility:

In [31]:
import torch
import torch.nn as nn
import random

def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)

set_seed(CONFIG['seed'])

Generate dataset:

In [32]:
import random
import torch
import itertools
from sklearn.model_selection import train_test_split

def generate_unique_data(sequence_length):
    # Generate all possible unique binary sequences
    all_sequences = list(itertools.product([0, 1], repeat=sequence_length))
    data = []
    labels = []

    for seq in all_sequences:
        parity = sum(seq) % 2
        data.append(seq)
        labels.append(parity)

    return torch.tensor(data, dtype=torch.float32).unsqueeze(-1), torch.tensor(labels)

# Config
seed = CONFIG["seed"]
val_size = CONFIG["val_size"]
sequence_length = CONFIG["sequence_length"]

# 1. Generate all unique data
X, Y = generate_unique_data(sequence_length)

# 2. Shuffle the data before splitting
torch.manual_seed(seed)  # ensure reproducibility
perm = torch.randperm(len(X))

X = X[perm]
Y = Y[perm]

# 4. Now use train_test_split
X_train, X_val, Y_train, Y_val = train_test_split(
    X, Y, test_size=val_size, random_state=seed, shuffle=True
)


X_train.shape, Y_train.shape, X_val.shape, Y_val.shape

(torch.Size([819, 10, 1]),
 torch.Size([819]),
 torch.Size([205, 10, 1]),
 torch.Size([205]))

In [33]:
from torch.utils.data import TensorDataset, DataLoader

# Create TensorDataset
train_dataset = TensorDataset(X_train, Y_train)
val_dataset = TensorDataset(X_val, Y_val)

# Create DataLoader
batch_size = CONFIG['batch_size']
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# Sample a batch (iterator)
X_batch, Y_batch = next(iter(train_loader))
X_batch.shape, Y_batch.shape

(torch.Size([32, 10, 1]), torch.Size([32]))

Build the model:

In [34]:
def build_model():
    input_size = CONFIG["input_size"]
    hidden_size = CONFIG["hidden_size"]
    output_size = CONFIG["output_size"]

    # Parameters
    i2h_weights = nn.Parameter(torch.empty(input_size + hidden_size, hidden_size))
    i2h_bias = nn.Parameter(torch.zeros(hidden_size))
    h2o_weights = nn.Parameter(torch.empty(hidden_size, output_size))
    h2o_bias = nn.Parameter(torch.zeros(output_size))

    # Init
    nonlinearity = CONFIG['nonlinearity']
    if CONFIG['weight_init'] == 'xavier':
        nn.init.xavier_uniform_(i2h_weights, gain=nn.init.calculate_gain(nonlinearity))
        nn.init.xavier_uniform_(h2o_weights, gain=nn.init.calculate_gain(nonlinearity))
    elif CONFIG['weight_init'] == 'kaiming':
        nn.init.kaiming_uniform_(i2h_weights, mode='fan_in', nonlinearity=nonlinearity)
        nn.init.kaiming_uniform_(h2o_weights, mode='fan_in', nonlinearity=nonlinearity)

    return [i2h_weights, i2h_bias, h2o_weights, h2o_bias]

model_params = build_model()
i2h_weights, i2h_bias, h2o_weights, h2o_bias = model_params

Test forward pass:

In [35]:
n_epochs = CONFIG['n_epochs']
hidden_size = CONFIG['hidden_size']
learning_rate = CONFIG['learning_rate']

# Activation function
def activation(x):
    if CONFIG['nonlinearity'] == 'relu': return torch.relu(x)
    elif CONFIG['nonlinearity'] == 'sigmoid': return torch.sigmoid(x)
    elif CONFIG['nonlinearity'] == 'tanh': return torch.tanh(x)
    else: raise ValueError(f"Unknown activation function: {CONFIG['nonlinearity']}")

# Forward pass
def forward(sequence):
    hidden_size = CONFIG['hidden_size']
    h = torch.zeros(sequence.size(0), hidden_size)
    for t in range(sequence.size(1)):
        combined = torch.cat((sequence[:, t], h), dim=1)
        h = activation(combined @ i2h_weights + i2h_bias)
    output = h @ h2o_weights + h2o_bias
    return output

In PyTorch, nn.CrossEntropyLoss expects:

Predictions: raw scores (logits), shape [batch_size, num_classes].

Targets: class indices (not one-hot), shape [batch_size], where each entry is the correct class index.

In [36]:
import torch
import torch.nn as nn

# Example: batch of 3 samples, 4 classes
logits = torch.tensor([[9.0, 0.5, 0.3, 0.2],
                       [0.1, 2.1, 0.3, 0.1],
                       [0.2, 0.2, 0.2, 1.5]])  # raw outputs

targets = torch.tensor([0, 1, 3])  # correct class indices

loss_fn = nn.CrossEntropyLoss()
loss = loss_fn(logits, targets)

print(loss)

tensor(0.3200)


In [37]:
criterion = nn.CrossEntropyLoss()
index, (X_batch, Y_batch) = next(enumerate(train_loader))
X_batch.shape, Y_batch.shape
outputs = forward(X_batch)
loss = criterion(outputs, Y_batch)
loss

tensor(0.9736, grad_fn=<NllLossBackward0>)

Train:

In [38]:
from tqdm import tqdm
import torch
import torch.nn as nn

def train():
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model_params, lr=CONFIG['learning_rate'])

    avg_train_losses, avg_val_losses = [], []
    avg_train_accs, avg_val_accs = [], []

    total_steps = CONFIG['n_epochs'] * (len(train_loader) + len(val_loader))

    with tqdm(total=total_steps, dynamic_ncols=True) as pbar:
        for epoch in range(CONFIG['n_epochs']):
            # Training phase
            train_losses, train_accs = [], []
            #model.train()

            for X_batch, Y_batch in train_loader:
                outputs = forward(X_batch)
                loss = criterion(outputs, Y_batch)

                optimizer.zero_grad()
                loss.backward()

                if CONFIG.get('max_grad_norm'):
                    nn.utils.clip_grad_norm_(model_params, CONFIG['max_grad_norm'])

                optimizer.step()

                preds = outputs.argmax(dim=1)
                acc = (preds == Y_batch).float().mean()

                train_losses.append(loss.item())
                train_accs.append(acc.item())

                # Progress bar update (no postfix override)
                pbar.update(1)

            # Validation phase
            val_losses, val_accs = [], []
            #model.eval()

            with torch.no_grad():
                for X_batch, Y_batch in val_loader:
                    outputs = forward(X_batch)
                    loss = criterion(outputs, Y_batch)

                    preds = outputs.argmax(dim=1)
                    acc = (preds == Y_batch).float().mean()

                    val_losses.append(loss.item())
                    val_accs.append(acc.item())

                    # Progress bar update (no postfix override)
                    pbar.update(1)

            # Aggregate metrics
            avg_train_loss = sum(train_losses) / len(train_losses)
            avg_train_acc = sum(train_accs) / len(train_accs)
            avg_val_loss = sum(val_losses) / len(val_losses)
            avg_val_acc = sum(val_accs) / len(val_accs)

            avg_train_losses.append(avg_train_loss)
            avg_val_losses.append(avg_val_loss)
            avg_train_accs.append(avg_train_acc)
            avg_val_accs.append(avg_val_acc)

            # Final summary for the epoch
            pbar.set_postfix({
                'Epoch': epoch + 1,
                'Train Loss': f"{avg_train_loss:.4f}",
                'Train Acc': f"{avg_train_acc:.4f}",
                'Val Loss': f"{avg_val_loss:.4f}",
                'Val Acc': f"{avg_val_acc:.4f}"
            })

# Run training
train()

 15%|█▌        | 5044/33000 [00:19<01:47, 258.95it/s, Epoch=152, Train Loss=0.6753, Train Acc=0.5715, Val Loss=0.9530, Val Acc=0.2603]


KeyboardInterrupt: 

Evaluate:

In [ ]:
def eval():
    test_data, test_labels = generate_data(10, CONFIG['sequence_length'])
    with torch.no_grad():
        outputs = forward(test_data)
        predicted = torch.argmax(outputs, dim=1)
        accuracy = (predicted == test_labels).float().mean()

        print("\nTest Results:")
        for seq, pred, label in zip(test_data.squeeze(-1), predicted, test_labels):
            print(f"Sequence: {seq.tolist()}, Predicted: {pred.item()}, Actual: {label.item()}")

        print(f"\nFinal Test Accuracy: {accuracy.item():.4f}")

eval()